## 📝 XGBoostClassifier
---
### Gök Cisimlerinin Sınıflandırılması (SDSS): XGBoost ile Çok Sınıflı Sınıflandırma
Bu projede, Sloan Digital Sky Survey (SDSS) veritabanından alınan astronomik gözlem verileri kullanılarak uzaydaki gök cisimlerinin Galaksi (Galaxy), Yıldız (Star) veya Kuasar (QSO) olup olmadığı sınıflandırılmıştır. Yüksek boyutlu ve karmaşık yapıdaki bu veriyi modellemek için, Gradient Boosting algoritmalarının en güçlü temsilcilerinden biri olan XGBoost (Extreme Gradient Boosting) kullanılmış ve hiperparametre optimizasyonu (GridSearchCV) ile model performansı maksimize edilmiştir.

---

### 💻 Kütüphaneler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Görselleştirme Ayarları
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100

# Makine Öğrenimi: Ön İşleme ve Değerlendirme
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Makine Öğrenimi: Model
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

### 📝 Veri Yükleme ve Temizlik
---
#### 2. Veri Yükleme ve Temizleme
SDSS veri setinde modelin öğrenmesine katkı sağlamayacak olan tanımlayıcı ID'ler ve donanımsal okuma kodları (objid, specobjid, run, rerun, camcol, field) veri setinden çıkarılmıştır. Bu işlem, modelin gereksiz gürültüden (noise) arındırılmasını sağlar.

In [ ]:
#https://www.kaggle.com/datasets/camnugent/california-housing-prices/data

In [ ]:
df = pd.read_csv("20-digitalskysurvey.csv")

In [ ]:
df.head()

In [ ]:
# Anlamsız metadata ve ID kolonlarının düşürülmesi
columns_to_drop = ['objid', 'specobjid', 'rerun', 'camcol', 'field', 'run']
df.drop(columns_to_drop , axis=1,inplace=True)

In [ ]:
df.info()

In [ ]:
print(df['class'].value_counts())

###📝 Keşifsel Veri Analizi
---
#### 3. Keşifsel Veri Analizi (EDA)
Farklı gök cisimlerinin astrofiziksel özelliklerini (özellikle uzaydaki **uzaklık/hız** ilişkisini gösteren redshift yani kırmızıya kayma oranını) analiz ediyoruz.

### 💻 EDA Görselleştirmeleri

In [ ]:
# 1. Redshift'e Karşı Diğer Özelliklerin Dağılımı (Scatter Plots)
fig, axes = plt.subplots(1,3, figsize=(18,5))

sns.scatterplot(data = df , x = 'redshift' , y = 'ra' , hue = 'class',alpha = 0.7, ax = axes[0])
axes[0].set_title("Kırmızıya Kayma (Redshift) vs RA")

sns.scatterplot(data = df , x = "redshift" , y = "dec" , hue = "class" , alpha = 0.7 , ax = axes[1])
axes[1].set_title("Kırmızıya Kayma (Redshift) vs DEC")

sns.scatterplot(data = df , x = "redshift" , y = "plate" , hue = "class" , alpha = 0.7 , ax = axes[2])
axes[2].set_title("Kırmızıya Kayma (Redshift) vs Plate")

plt.tight_layout()
plt.show()

In [ ]:
# 2. Sınıflara Göre Redshift Yoğunluğu (Histograms)
fig , axes = plt.subplots(nrows=1, ncols=3 ,figsize=(18,5), sharey=True)

sns.histplot(df[df["class"] == "GALAXY"]["redshift"], color="blue", kde=True, ax=axes[0])
axes[0].set_title("Galaksi Redshift Dağılımı")

sns.histplot(df[df["class"] == "QSO"]["redshift"], color="green", kde=True, ax=axes[1])
axes[1].set_title("Kuasar (QSO) Redshift Dağılımı")

sns.histplot(df[df["class"] == "STAR"]["redshift"], color="orange", kde=True, ax=axes[2])
axes[2].set_title("Yıldız (Star) Redshift Dağılımı")

plt.tight_layout()
plt.show()

### 📝 Veri Ön İşleme
---
#### 4. Veri Hazırlığı ve Ön İşleme
Makine öğrenimi modelleri metinsel hedef değişkenleri anlayamadığı için LabelEncoder kullanarak sınıfları sayısallaştırıyoruz.
(Not: GALAXY = 0, QSO = 1, STAR = 2 olarak kodlanacaktır).

💡**Mühendislik Notu (Scaling ve XGBoost) :**
Mesafe tabanlı algoritmaların **(KNN, SVM, Lojistik Regresyon)** aksine, **XGBoost** gibi ağaç tabanlı algoritmalar özellik ölçeklendirmesine **(StandardScaler)** ihtiyaç duymaz. Ağaçlar veriyi belirli eşik değerlerinden böldüğü için değişkenlerin kendi içindeki ölçekleri sonucu etkilemez. Yine de Pipeline standartlarını korumak adına ölçeklendirme işlemi projede tutulmuştur.

---

### 💻 Encoding ve Split

In [ ]:
# Hedef değişkeni sayısallaştırma (Encoding)
le = LabelEncoder()

In [ ]:
df["class"] = le.fit_transform(df["class"])

In [ ]:
# Hangi sınıfın hangi sayıya atandığını kontrol edelim
print("Sınıf Eşleşmeleri:", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Bağımsız (X) ve Bağımlı (y) Değişkenler
X = df.drop("class",axis=1)
y = df["class"]

In [ ]:
# Train / Test Bölünmesi (Verinin 3'te 1'i test için ayrıldı)
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size= 0.33, random_state=42, stratify=y)

In [ ]:
# Ölçeklendirme (XGBoost için zorunlu olmasa da best-practice olarak eklendi)
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 📝 Baseline XGBoost
---

#### 5. Varsayılan (Baseline) XGBoost Modeli Eğitimi
Öncelikle hiçbir hiperparametre ayarı yapmadan, algoritmanın varsayılan (default) yeteneklerini görmek için temel bir model eğitiyoruz.

### 💻 Baseline Model

In [ ]:
# Baseline Model Eğitimi
xgb_base = XGBClassifier(n_estimators = 100 ,random_state = 42)
xgb_base.fit(X_train_scaled, y_train)

In [ ]:
# Tahmin
y_pred_base = xgb_base.predict(X_test_scaled)

In [ ]:
# Değerlendirme (Gerçek Y, Tahmin Edilen Y sırasına dikkat ediniz!)
print("=== XGBoost (Baseline Model) Performansı ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_base):.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred_base, target_names=le.classes_))

---

### 📝 Hiperparametre Optimizasyonu
---
#### 6. Hiperparametre Optimizasyonu (GridSearchCV)
**XGBoost**, düzinelerce parametre barındıran oldukça kompleks bir yapıdır. Aşırı öğrenmeyi **(Overfitting)** engellemek ve performansı artırmak için en kritik parametreleri bir arama uzayına sokuyoruz:

* **n_estimators:** Kurulacak karar ağacı sayısı.
* **learning_rate:** Modelin öğrenme hızı (adımların büyüklüğü).
* **max_depth:** Ağaçların ne kadar derine inebileceği (Overfit kontrolü).
* **colsample_bytree:** Her ağaç kurulurken kullanılacak özelliklerin rastgele seçilme oranı.

### 💻 GridSearchCV ve Final Model

In [ ]:
# Test edilecek parametre kombinasyonları
params = {
    "n_estimators" : [100,200,300],      # Maliyeti düşürmek için 500'ü çıkardık
    "learning_rate" : [0.01, 0.1],  
    "max_depth" : [5,8,12],              # Çok derin ağaçlar ezbere yol açar
    "colsample_bytree": [0.5, 0.8, 1.0]
}


print("GridSearchCV ile en iyi parametreler aranıyor (Bu işlem donanıma bağlı olarak birkaç dakika sürebilir)...\n")


# Grid Search Kurulumu
grid = GridSearchCV(
    estimator=XGBClassifier(random_state = 42),
    param_grid= params,
    cv = 3,        # İşlem süresini makul tutmak için 3 Fold
    n_jobs= -1,    # CPU'nun tüm çekirdeklerini kullan
    verbose = 1 
)

grid.fit(X_train_scaled , y_train)

# En iyi parametreleri yazdırma
print(f"Bulunan En İyi Parametreler: {grid.best_params_}")

---
### 📝 Nihai Değerlendirme
---
#### 7. Optimize Edilmiş Final Modelin Değerlendirilmesi
Grid Search sonucunda elde ettiğimiz en başarılı model ile test veri setimiz üzerinde nihai tahminlerimizi yapıyor ve endüstri standardı bir ısı haritası **(Heatmap)** ile Karmaşıklık Matrisini inceliyoruz.

### 💻 Final Evaluation

In [ ]:
# En iyi model ile tahmin yapma
y_pred_tuned = grid.predict(X_test_scaled)

print("=== XGBoost (Optimize Edilmiş Model) Performansı ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_tuned):.4f}\n")
print("Classification Report:\n", classification_report(y_test, y_pred_tuned, target_names=le.classes_))

# Endüstri Standardı Confusion Matrix Görselleştirmesi
cm = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, 
            yticklabels=le.classes_)
plt.title('Final Model Karmaşıklık Matrisi (Confusion Matrix)', fontsize=14, fontweight='bold')
plt.ylabel('Gerçek Sınıflar (True Label)')
plt.xlabel('Tahmin Edilen Sınıflar (Predicted Label)')
plt.show()